## Advanced Python for NLP (2026)

**Rainer Osswald, Yulia Zinova**

# Topic 9(2): The pandas library

This notebook draws heavily on [McKinney (2018)](https://wesmckinney.com/pages/book.html) and [VanderPlas (2016)](https://jakevdp.github.io/PythonDataScienceHandbook/) without acknowledging this every time.

## Background

The creation of the pandas library was initiated by Wes McKinney in 2008 when he worked for a quantitative investment management firm.
The name _pandas_ derives from _panel data_, an econometric term for multidimensional structured datasets.
McKinney had the following requirements in mind ([McKinney 2018:5](https://wesmckinney.com/pages/book.html)):

* Data structures with labeled axes supporting automatic or explicit data alignment
* Integrated time series functionality
* Arithmetic operations and reductions that preserve metadata
* Flexible handling of missing data
* Merge and other relational operations found in popular databases

## Introduction to pandas data structures

The pandas library is addressed by the identifier `pandas` and is commonly imported under the alias `pd`.
Pandas has two central data structures, **Series** and **DataFrame**.

### Series

A `Series` object is similar to a 1-dimensional NumPy array together with an associated array of data labels, called the **index** of the Series.
If the index is not explicitly specified, a standard range of integers is chosen.

In [ ]:
import numpy as np
import pandas as pd
data = pd.Series([9, 3, 5, 7])

Series have the attributes `values`, `index`, `size` and `dtype`, among others.
A specified index can be used like a range index for accessing and setting the values of the Series.
The index can contain only **hashable** elements and must have the same size as the Series.

In [ ]:
data = pd.Series([9, 3, 5, 7], index = ['a', 'b', 'c', 'd'])
data['b'] = 4
data[['b', 'd']].values  # ⇒ array([4, 7])

Pandas Series (with explicit index) look similar to dictionaries in appearance but are much more efficiently implemented since the values form a NumPy array.
You can use a dictionary to initialize a pandas `Series` object and if you don't specify the index, the keys of the dictionary will be used as the index:

In [ ]:
dct = {'a': 9, 'b': 3, 'c': 5, 'd': 7}
data = pd.Series(dct)
'a' in data  # ⇒ True
data.index   # ⇒ Index(['a', 'b', 'c', 'd'], dtype='object')

You can overwrite the order of the original dictionary keys.
You can also add and remove index elements and even modify thereby the size of the resulting `Series` object:

In [ ]:
data = pd.Series(dct, index = ['e', 'd', 'c'])
data.size  # ⇒ 3
data['e']  # ⇒ NaN
data['d':'c']

The value `NaN` ("not a number") is used in pandas to mark missing values.
You can check for missing values with the function (or method) `isnull`/`notnull` which returns `True`/`False` if the value is missing for the respective index element, and `False`/`True` otherwise: 

In [ ]:
data.isnull()
data.notnull()

Note that the implicit Python-style indexing is always possible even if you provide an explicit index.
This can lead to confusion if your self-defined index is also based on natural numbers.
`Series` objects have the two attributes `loc` and `iloc` that provide two views on the object corresponding to the different indexations and slicings:
`iloc` is for the implicit Python-style index, `loc` is for an explicitly introduced index.
For the above example, this means that , e.g., `data.loc['d'] == data.iloc[1]` evaluates to `True`.

In [ ]:
data['d'] == data.loc['d']  # ⇒ True

### DataFrame

A `DataFrame` object can be thought of as a dictionary of `Series` objects that share the same index.
The Series are the (ordered) columns of the DataFrame.
They may have different data types.
A DataFrame has both a row and a column index.
Internally, the data are stored in one or more two-dimensional blocks in memory.
One of the most common ways of creating a DataFrame is by starting with a dictionary of equal-length lists or 1-dimensional arrays:

In [ ]:
data = {'state': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada', 'Nevada'],
        'year': [2000, 2001, 2002, 2001, 2002, 2003],
        'pop': [1.5, 1.7, 3.6, 2.4, 2.9, 3.2]}
pd.DataFrame(data)

During creation, you can reorder and modify the column index given by the dictionary keys, and you can add an explicit row index:

In [ ]:
frame = pd.DataFrame(data,
                     columns=['year', 'state', 'pop', 'debt'],
                     index=['one', 'two', 'three', 'four', 'five', 'six'])
#frame['state']

Rows can be accessed via the `loc` or the `iloc` attribute:

In [ ]:
frame.iloc[1]  # equivalent: frame.loc['two']

Columns can be accessed by their index or, alternatively, as attributes of the DataFrame.
That is, `frame['state']` is equivalent to `frame.state`.
**Remark:** Columns returned from indexing are **views** on the data, **not copies**.

Columns can be modified by assignment.
The following examples illustrate different ways of how to do this:

In [ ]:
frame['debt'] = 16.5     # set all 'debt' values to the same scalar value
frame['debt'] = np.linspace(10.2, 18.6, 6)

Pandas DataFrames can easily cope with missing data:

In [ ]:
val = pd.Series([-1.2, -1.5, -1.7], index=['two', 'four', 'five'])
frame['debt'] = val

There are other ways of creating a DataFrame.
For instance, you can start with a dictionary of Series or from a 2-dimensional NumPy array with column (and row) indexes added.
The latter option is sketched by the following snippet:

In [ ]:
pd.DataFrame(np.random.rand(3, 2),
             columns=['first', 'second'],
             index=['a', 'b', 'c'])

## Basic functionality

### Reindexing and index dropping

An important method is `reindex`, which means to rearrange the data according to a new index, with missing value indicators added if necessary.
The following example applies `reindex` to a `Series` object (compare this with a similar example discussed above):

In [ ]:
dct = {'a': 9, 'b': 3, 'c': 5, 'd': 7}
data = pd.Series(dct)
data.reindex(['e', 'd', 'c'])

With `DataFrame` objects, `reindex` can either change the row index, the columns, or both:

In [ ]:
frame = pd.DataFrame(np.arange(9).reshape((3, 3)),
                     index=['a', 'c', 'd'],
                     columns=['Ohio', 'Texas', 'California'])
frame.reindex(['a', 'b', 'c', 'd'], columns=['Texas', 'Utah', 'California'])

The method `drop` allows you to remove index values from one of the axis:

In [ ]:
frame.drop(['a'])  # equivalent: frame.drop(['a'], axis='rows')
frame.drop(['Ohio', 'Texas'], axis='columns')

Many functions which, like `drop`, modify the size or the shape of a Series or DataFrame, can be used destructively **in-place** by setting the parameter `inplace=True`.

### Indexing, slicing and filtering

Indexing, slicing and filtering of DataFrames works basically in the same way as with NumPy arrays.
But notice that slicing with labels behaves slightly different insofar as the end point is included:

In [ ]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=['Ohio', 'Colorado', 'Utah', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
data['two']
data[['three', 'one']]
data[data < 10]
data.loc[:'Utah', 'three':'one':-1]  # equivalent: data.iloc[:3, 2::-1]

### Data alignment and vectorized computation

Pandas allows you to apply operations to Series and DataFrames with differing indexes.
Missing values will be introduced automatically for locations where the two indexes do not overlap.
You can specify a `fill_value` that applies to all locations where at most one value is missing.
Consider the following example:

In [ ]:
df1 = pd.DataFrame(np.arange(12.).reshape((3, 4)),
                   index=list('xyz'),
                   columns=list('abcd'))
df2 = pd.DataFrame(np.arange(20.).reshape((4, 5)),
                   index=list('xyvw'),
                   columns=list('fedcb'))
df1 + df2
df1.add(df2, fill_value=0)

If you want to apply an aggregating function such as `min` of `max` to each row or column of a DataFrame, you can do that by the `apply` method:

In [ ]:
frame = pd.DataFrame(np.random.randn(4, 3),
                     columns=list('bde'),
                     index=['Utah', 'Ohio', 'Texas', 'Oregon'])
f = lambda x: x.max() - x.min()
frame.apply(f)
frame.apply(f, axis='columns')

### Sorting

You can sort the values of a Series or DataFrame by index or by value:

In [ ]:
frame = pd.DataFrame(np.arange(8).reshape((2, 4)),
                     index=['three', 'one'],
                     columns=['d', 'a', 'b', 'c'])
#frame.sort_index()
#frame.sort_index(axis=1)
#frame.sort_index(axis=1, ascending=False)

In [ ]:
frame.sort_values(by='b', ascending=False)

## Combining datasets

### Hierarchical indexing

Hierarchical indexing allows you to have **multiple index levels** on an axis.
The basic idea is that indexes can have a higher dimension than one.
This is illustrated by the following example of a Series.
Without hierarchical indexing you may feel the need to structure your index yourself:

In [ ]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)

Hierarchical indexing by pandas' `MultiIndex` objects allows you to do this more systematically.

In [ ]:
index = pd.MultiIndex.from_tuples(index)
pop = pop.reindex(index)
#pop[:, 2010]

Hierarchical indexing provides also a useful way to rearrange the data in a DataFrame.
There are two methods for this:
`stack`, which rotates or pivots from the columns to the rows, and `unstack`, which does the reverse.
Here is an example:

In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'], name='number'))
display(data.stack())

The most useful thing about multi-indexing is that it provides us with a straightforward way to represent arrays of more than two dimensions.
Just as we were able to use multi-indexing to represent two-dimensional data within a one-dimensional Series, it allows us to represent data of three or more dimensions in a Series or DataFrame. 

In [ ]:
pop_18 = pd.DataFrame({'total': pop,
                       'under18': [9267089, 9284094,
                                   4687374, 4318033,
                                   5906301, 6879014]})
#display(pop_18)
#pop_18.unstack()

### Merging datasets

You may be familiar with merging tables in relational databases by join operations.
Pandas has a `merge` function for DataFrames, whose effect is illustrated by the following example:

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})

In [ ]:
pd.merge(df1, df2)
pd.merge(df1, df2, on='key')

If not otherwise specified, `merge` will use the overlapping column names as keys.
It is nevertheless good practice to specify the column to join on explicitly.
By default, `merge` creates an **inner join**.
That is, the keys of the result are the intersection of those that are present in both tables.
If you want to create an **outer join** instead, you need to specify `how='outer'`:

In [ ]:
pd.merge(df1, df2, how='outer')

If you want to merge with multiple keys, you need to pass them as a list of column names:

In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

You can also merge by using the index of a DataFrame as key by specifying `left_index=True` and/or `right_index=True`.

### Concatenation along an axis

Remember that NumPy provides a `concatenate` function for binding together arrays along an axis.
You can do a similar thing with Series and DataFrames.
For instance, you can concatenate several Series to a DataFrame:

In [ ]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['b', 'c', 'd'])
s3 = pd.Series([5, 6], index=['d', 'e'])
pd.concat([s1, s2, s3], axis=1)

## Plotting with pandas

Pandas comes with integrated visualization functionality.
(Nevertheless, you should consider the libraries `mathplotlib` and `seaborn` as well.)
Here are two simple examples:

In [ ]:
df = pd.DataFrame(np.random.randn(11, 4).cumsum(0),
                  columns=['A', 'B', 'C', 'D'],
                  index=np.linspace(0, 100, 11))
#df.plot()

In [ ]:
df = pd.DataFrame(np.random.rand(6, 4),
                  index=['one', 'two', 'three', 'four', 'five', 'six'],
                  columns=pd.Index(['A', 'B', 'C', 'D'], name='Genus'))
#df.plot.bar()

## Conclusion
This notebook gave a brief introduction into a few of the basic functionalities of the pandas library.
An important topic left out is the availability of various built-in functions for computing descriptive statistics on the data.

## References

* [Official pandas documentation](https://pandas.pydata.org/pandas-docs/stable/)
* [Wes McKinney (2018). _Python for Data Analysis_. 2nd edition. O'Reilly.](https://wesmckinney.com/pages/book.html)
* [Jake VanderPlas (2016). _Python Data Science Handbook_. O'Reilly. (Freely available, including Jupyter notebooks!)](https://jakevdp.github.io/PythonDataScienceHandbook/)